In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [2]:
df = pd.read_csv("../data/processed/clean_housing.csv")

In [3]:
X = df.drop(columns=["PRICE"])
y = np.log1p(df["PRICE"])

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
print(X_train.shape)
print(y_train.shape)

(2133, 21)
(2133,)


In [6]:
numeric_features = [
    "LAND_AREA",
    "ROAD_ACCESS",
    "FLOOR",
    "BEDROOM",
    "BATHROOM",
    "BUILT_YEAR",
    "AMENITY_COUNT",
    "bedroom_density",
    "bath_per_bed"
]

categorical_features = [
    "FACING",
    "LOCATION"
]

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder='passthrough'
)

In [8]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [9]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(max_iter=10000),
    "Ridge": Ridge(max_iter=10000),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(),
    "AdaBoost Regressor": AdaBoostRegressor(),
    "XGBRegressor": XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8
    ), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
}

In [10]:
for name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    # Make predictions
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)
    
    # Evaluate Train and Test dataset
    train_mae, train_rmse, train_r2 = evaluate_model(y_train, y_train_pred)
    test_mae, test_rmse, test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(name)
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(train_mae))
    print("- R2 Score: {:.4f}".format(train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(test_mae))
    print("- R2 Score: {:.4f}".format(test_r2))
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 0.5206
- Mean Absolute Error: 0.3262
- R2 Score: 0.2847
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 0.5219
- Mean Absolute Error: 0.3403
- R2 Score: 0.2582


Lasso
Model performance for Training set
- Root Mean Squared Error: 0.6156
- Mean Absolute Error: 0.4218
- R2 Score: 0.0000
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 0.6069
- Mean Absolute Error: 0.4262
- R2 Score: -0.0031


Ridge
Model performance for Training set
- Root Mean Squared Error: 0.5207
- Mean Absolute Error: 0.3262
- R2 Score: 0.2846
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 0.5216
- Mean Absolute Error: 0.3401
- R2 Score: 0.2591


Random Forest Regressor
Model performance for Training set
- Root Mean Squared Error: 0.3650
- Mean Absolute Error: 0.1871
- R2 Score: 0.6484
--------------------